<a href="https://colab.research.google.com/github/Innovatewithapple/TransformersProjects/blob/main/VITAdvance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import tensorflow as tf
from tensorflow.keras import layers

In [24]:
#Transformer_Block
class TransformerBlock(layers.Layer):
  def __init__(self,num_heads,embed_dim,ff_dim,rate=0.1):
    super().__init__()

    self.att = layers.MultiHeadAttention(num_heads=num_heads,key_dim=embed_dim)

    self.fft = tf.keras.Sequential([
        layers.Dense(ff_dim,activation='relu'),
        layers.Dense(embed_dim)
    ])

    self.layerNorm1 = layers.LayerNormalization(epsilon=1e-6)
    self.layerNorm2 = layers.LayerNormalization(epsilon=1e-6)

    self.Dropout1 = layers.Dropout(rate=rate)
    self.Dropout2 = layers.Dropout(rate=rate)

  def call(self,inputs,mask=None):
    attention_output = self.att(query=inputs,key=inputs,value=inputs,attention_mask=mask)
    attention_output = self.Dropout1(attention_output)

    output1 = self.layerNorm1(inputs + attention_output)

    ff_output = self.fft(output1)
    ff_output = self.Dropout2(ff_output)
    output2 = self.layerNorm2(output1 + ff_output)
    return output2

In [25]:
#Patch Embedding
class PatchEmbedding(layers.Layer):
  def __init__(self,embed_dim,image_size,patch_size):
    super().__init__()
    self.num_patchs = (image_size // patch_size) ** 2
    self.embed_dim = embed_dim

    self.patchEmbed = layers.Dense(embed_dim)
    self.positionEmbed = layers.Embedding(input_dim=self.num_patchs,output_dim=embed_dim)

  def call(self,patchs):
    position_range = tf.range(start=0,limit=self.num_patchs,delta=1)

    self.position = self.positionEmbed(position_range)
    self.patchoutput = self.patchEmbed(patchs)
    return self.patchoutput + self.position

In [26]:
#Vision transfomer
class VisionTransformer(tf.keras.Model):
  def __init__(self,embed_dim,imagesize,patch_size,num_heads,ff_dim,num_layers,num_classes):
    super().__init__()
    self.patchEmbed = PatchEmbedding(embed_dim=embed_dim,image_size=imagesize,patch_size=patch_size)

    self.transformer_Layer = [
        TransformerBlock(num_heads=num_heads,embed_dim=embed_dim,ff_dim=ff_dim)
        for _ in range(num_layers)
        ]

    self.flatten = layers.Flatten()
    self.dense = layers.Dense(num_classes,activation='softmax')

  def call(self,images):
    patches = self.extract_patchs(images)
    x = self.patchEmbed(patches)

    for transformer_layer in self.transformer_Layer:
      x = transformer_layer(x)

    x = self.flatten(x)
    return self.dense(x)

  def extract_patchs(self,images):
    batch_size = tf.shape(images)[0]

    patches = tf.image.extract_patches(
        images=images,
        sizes=[1,patch_size,patch_size,1],
        strides=[1,patch_size,patch_size,1],
        rates=[1,1,1,1], # its for skipping the pixals, 1,1,1,1 means take each pixel, if we use 2,2,2,2 it pick every second pixel
        padding='VALID' #Only cut if the square fits perfectly. we use 'SAME' also but it will add black pixels if image image has weird size and cant cut perfect square
      )
    patches = tf.reshape(patches,[batch_size,-1,patch_size * patch_size * 3])
    return patches

In [27]:
embedding_dim = 128
num_heads = 4
ff_dim = 512
num_layers = 6
num_classes = 10  # CIFAR-10
image_size = 224
patch_size = 16

vit = VisionTransformer(
    embed_dim=embedding_dim,
    imagesize=image_size,
    patch_size=patch_size,
    num_heads=num_heads,
    ff_dim=ff_dim,
    num_layers=num_layers,
    num_classes=num_classes
)

images = tf.random.uniform((32, 224, 224, 3))  # Batch of 32 images
output = vit(images)

print(output.shape)  # Expected: (32, 10)

(32, 10)
